In [0]:
from pyspark.sql import functions as F

In [0]:
#Configure the dataset
SEED = 20261022
EVENT_COUNT = 10000
CUSTOMER_COUNT = 1000
PRODUCT_COUNT = 20

print("Random Seed:", SEED)
print("Engagement Events:", EVENT_COUNT)

In [0]:
#Generate engagement events
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.range(1, EVENT_COUNT + 1)

# Generate unique event IDs
events = events.withColumn(
    "event_id",
    F.concat(
        F.lit("EVN-2026-"),
        F.lpad(F.col("id").cast("string"), 8, "0")
    )
)

# Generate deterministic customer mapping
customer_lookup = (
    spark.table("customer_master")
    .select("customer_id")
    .withColumn(
        "customer_number",
        F.row_number().over(
            Window.orderBy("customer_id")
        )
    )
)

events = events.withColumn(
    "customer_number",
    (
        F.pmod(
            F.xxhash64("id", F.lit(SEED)),
            F.lit(CUSTOMER_COUNT)
        ) + 1
    ).cast("int")
)

events = events.join(
    customer_lookup,
    on="customer_number",
    how="inner"
)

# Generate event types
event_hash = F.pmod(
    F.xxhash64("id", F.lit("event")),
    F.lit(100)
)

events = events.withColumn(
    "event_type",
    F.when(event_hash < 30, "WEBSITE_VISIT")
     .when(event_hash < 55, "PRODUCT_VIEW")
     .when(event_hash < 70, "ADD_TO_CART")
     .when(event_hash < 82, "EMAIL_OPEN")
     .when(event_hash < 90, "EMAIL_CLICK")
     .when(event_hash < 96, "PURCHASE")
     .otherwise("SUPPORT_PAGE_VIEW")
)

# Generate channels
events = events.withColumn(
    "channel",
    F.when(F.col("event_type").isin("WEBSITE_VISIT", "PRODUCT_VIEW", "ADD_TO_CART"), "WEBSITE")
     .when(F.col("event_type").isin("EMAIL_OPEN", "EMAIL_CLICK"), "EMAIL")
     .when(F.col("event_type") == "PURCHASE", "MOBILE_APP")
     .otherwise("SUPPORT_PORTAL")
)

# Generate product IDs for product-related events
product_lookup = (
    spark.table("product_master")
    .select("product_id")
    .withColumn(
        "product_number",
        F.row_number().over(Window.orderBy("product_id"))
    )
)

events = events.withColumn(
    "product_number",
    (
        F.pmod(
            F.xxhash64("id", F.lit("product")),
            F.lit(PRODUCT_COUNT)
        ) + 1
    ).cast("int")
)

events = events.join(
    product_lookup,
    on="product_number",
    how="left"
)

events = events.withColumn(
    "product_id",
    F.when(
        F.col("event_type").isin(
            "PRODUCT_VIEW", "ADD_TO_CART", "PURCHASE"
        ),
        F.col("product_id")
    ).otherwise(F.lit(None).cast("string"))
)

# Generate timestamps between January 2025 and October 2026
events = events.withColumn(
    "event_timestamp",
    F.expr(
        "timestamp_seconds(unix_timestamp('2025-01-01 00:00:00') + "
        "CAST(pmod(xxhash64(id, 'time'), 57600000) AS BIGINT))"
    )
)

# Generate campaign IDs for email interactions
events = events.withColumn(
    "campaign_id",
    F.when(
        F.col("event_type").isin("EMAIL_OPEN", "EMAIL_CLICK"),
        F.concat(
            F.lit("CMP-2026-"),
            F.lpad(
                (
                    F.pmod(
                        F.xxhash64("id", F.lit("campaign")),
                        F.lit(10)
                    ) + 1
                ).cast("string"),
                3,
                "0"
            )
        )
    ).otherwise(F.lit(None).cast("string"))
)

# Select final columns
engagement_df = events.select(
    "event_id",
    "customer_id",
    "event_type",
    "event_timestamp",
    "channel",
    "product_id",
    "campaign_id"
)

display(engagement_df.limit(10))

#####Validate engagement data


In [0]:
#A. Check total records
print("Total Events:", engagement_df.count())

In [0]:
#Check duplicate event IDs
engagement_df.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
#Check invalid customer IDs
invalid_customers = engagement_df.join(
    spark.table("customer_master").select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Invalid Customer IDs:", invalid_customers.count())

In [0]:
#Check invalid product IDs
invalid_products = (
    engagement_df
    .filter(F.col("product_id").isNotNull())
    .join(
        spark.table("product_master").select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print("Invalid Product IDs:", invalid_products.count())

In [0]:
#Analyze event distribution
engagement_df.groupBy("event_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
#Save the EngagementEvent table
engagement_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("engagement_event")

In [0]:
%sql
SELECT * FROM engagement_event LIMIT 20;